In [15]:
# 1. Open data and split month-wise
# Load all data
import json

with open("./data/real_world/all.json", "r") as f:
    all = json.load(f)

In [16]:
# Split into month-wise data
import pickle

month_files = []
for key, value in all.items():
    file_name = f"./data/real_world/months_raw/{key}.raw"
    month_files.append(file_name)
    # Only once to create the files
        
    if key == "20120201":
        with open(file_name, "wb") as f:
            pickle.dump(value,f)
            print(f"Created {file_name}")
    
    #with open(file_name, "wb") as f:
    #    pickle.dump(value, f)

Created ./data/real_world/months_raw/20120201.raw


In [5]:
# Get torch device
from local_utilities.gpu import get_gpu
device = get_gpu()

[nltk_data] Downloading package punkt_tab to /home/tobias/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
/home/tobias/.pyenv/versions/genai/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
from local_utilities import evaluate, generate_report
from local_utilities.dataset import tokenize_text

from transformers import logging
logging.disable_progress_bar()

In [7]:
# Initialize language detector
import torch
from transformers import pipeline, AutoModelForSequenceClassification, AutoTokenizer

def initialize_lang():
    model_name = "papluca/xlm-roberta-base-language-detection"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name)
    
    return tokenizer, model

In [8]:
# Classify language function
def classify_lang(tokenizer, model, text):
    inputs = tokenizer(text, padding=True, truncation=True, return_tensors="pt").to(device)

    with torch.no_grad():
        logits = model(**inputs).logits
        
    preds = torch.softmax(logits, dim=-1)
    
    id2lang = model.config.id2label
    vals, idxs = torch.max(preds,dim=1)

    return [id2lang[k.item()] for k, v in zip(idxs, vals)]

In [9]:
lang_tokenizer, lang_model = initialize_lang()
lang_model.to(device)

XLMRobertaForSequenceClassification LOAD REPORT from: papluca/xlm-roberta-base-language-detection
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


XLMRobertaForSequenceClassification(
  (classifier): XLMRobertaClassificationHead(
    (dense): Linear(in_features=768, out_features=768, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (out_proj): Linear(in_features=768, out_features=20, bias=True)
  )
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): L

In [10]:
# Encode for FFNN and SVM
from local_utilities.dataset import *
encoder = load_encoder()

def encode_paper(sentences):
    tokens = clean_token(sentences)
    encodings = encode_texts(tokens, device, encoder).to("cpu")
    return encodings

In [ ]:
# Iterate over papers
import os
import torch
from tqdm import tqdm
import traceback

from local_utilities.ffnn import *
from local_utilities.svm import *

for i in tqdm(range(len(month_files))):
    file = month_files[i] 
    en_files = 0
    other_langs = []
    # Load file
    try:
        # Load original data
        with open(file, "rb") as f:
            clean_data = pickle.load(f)
            if not "papers" in clean_data:
                print(f"{file} does not contain any data!!!")
                continue
            
        # Load already stored month data if it exists
        month_data = clean_data # Use original data as fallback if file does not exist yet
        stored_data_path = file + ".sbert.svm.ffnn"
        if os.path.exists(stored_data_path):
            with open(stored_data_path, "rb") as f:
                month_data = pickle.load(f)
                
                # Merge new 'clean' data into month data if there is new data present
                month_data["papers"] = clean_data["papers"] | month_data["papers"]                
            
        # Check if there are papers
        if not "papers" in month_data:
            print(f"{file} does not contain any data!!!")
            continue
                
        # Get individual papers
        for arxiv_identifier, paper_data in month_data["papers"].items():
            if not "sentences" in paper_data:
                print(f"No sentences found for {file} / {arxiv_identifier}")
                continue
            
            if len(paper_data["sentences"]) == 0:
                print(f"Empty sentences list found for {file} / {arxiv_identifier}")
                continue
            
            # Encode if not done already
            if not "sbert_encodings" in month_data["papers"][arxiv_identifier]:
                month_data["papers"][arxiv_identifier]["sbert_encodings"] = encode_paper(paper_data["sentences"])
                
            ## FFNN
            if not "ffnn" in month_data["papers"][arxiv_identifier]:
                embeddings = month_data["papers"][arxiv_identifier]["sbert_encodings"]
                results = check_ffnn(embeddings.to(device), device)
                month_data["papers"][arxiv_identifier]["ffnn"] = results
                torch.cuda.empty_cache()
            
            ## SVM    
            if not "svm" in month_data["papers"][arxiv_identifier]:
                embeddings = month_data["papers"][arxiv_identifier]["sbert_encodings"]
                svm_results = check_svm(embeddings.to(device))
                month_data["papers"][arxiv_identifier]["svm"] = svm_results
                
            ## Language detection
            if not "lang" in month_data["papers"][arxiv_identifier]:
                sentences = ". ".join(paper_data["sentences"])
                lang = classify_lang(lang_tokenizer, lang_model, sentences)
                month_data["papers"][arxiv_identifier]["lang"] = lang[0]
                if lang[0] == "en":
                    en_files += 1
                else:
                    other_langs.append(lang[0])
            else:
                if month_data["papers"][arxiv_identifier]["lang"] == "en":
                    en_files += 1
                else:
                    other_langs.append(month_data["papers"][arxiv_identifier]["lang"])
                
        # Store
        # Rename file
        file = file + ".sbert.svm.ffnn"
        with open(file, "wb") as f:
            pickle.dump(month_data, f)
        print(f"{file} contains {en_files} English documents, other languages are {set(other_langs)}")
    except Exception as e:
        print(f"Could not process {file}: {str(e)}")

  0%|          | 0/192 [00:00<?, ?it/s]

279


100%|██████████| 192/192 [00:09<00:00, 20.11it/s]

./data/real_world/months_raw/20120201.raw.sbert.svm.ffnn contains 266 English documents, other languages are ['hi', 'fr', 'de', 'hi', 'hi', 'hi', 'hi', 'hi', 'hi', 'hi', 'it', 'hi', 'de']


In [8]:
"""
# Iterate over files and generate FFNN results
import torch
from local_utilities.ffnn import *

for i in tqdm(range(len(month_files))):
    file = month_files[i]
    # Load file
    try:
        with open(file, "rb") as f:
            month_data = pickle.load(f)
            
        # Check if there are papers
        if not "papers" in month_data:
            print(f"{file} does not contain any data!!!")
            continue
                
        # Get individual papers
        for arxiv_identifier, paper_data in month_data["papers"].items():
            if not "sbert_encodings" in paper_data:
                print(f"No encodings found for {file} / {arxiv_identifier}")
                continue
                        
            if not "ffnn" in month_data["papers"][arxiv_identifier]:
                embeddings = month_data["papers"][arxiv_identifier]["sbert_encodings"]
                results = check_ffnn(embeddings.to(device), device)
                month_data["papers"][arxiv_identifier]["ffnn"] = results
                torch.cuda.empty_cache()

        # Store
        with open(file, "wb") as f:
            pickle.dump(month_data, f)
    except Exception as e:
        print(f"Could not process {file}: {str(e)}")
"""

'\n# Iterate over files and generate FFNN results\nimport torch\nfrom local_utilities.ffnn import *\n\nfor i in tqdm(range(len(month_files))):\n    file = month_files[i]\n    # Load file\n    try:\n        with open(file, "rb") as f:\n            month_data = pickle.load(f)\n\n        # Check if there are papers\n        if not "papers" in month_data:\n            print(f"{file} does not contain any data!!!")\n            continue\n\n        # Get individual papers\n        for arxiv_identifier, paper_data in month_data["papers"].items():\n            if not "sbert_encodings" in paper_data:\n                print(f"No encodings found for {file} / {arxiv_identifier}")\n                continue\n\n            if not "ffnn" in month_data["papers"][arxiv_identifier]:\n                embeddings = month_data["papers"][arxiv_identifier]["sbert_encodings"]\n                results = check_ffnn(embeddings.to(device), device)\n                month_data["papers"][arxiv_identifier]["ffnn"] = res

In [9]:
"""
# Iterate over files and generate SVM results
from local_utilities.svm import *

for i in tqdm(range(len(month_files))):
    file = month_files[i]
    # Load file
    try:
        with open(file, "rb") as f:
            month_data = pickle.load(f)
            
        # Check if there are papers
        if not "papers" in month_data:
            print(f"{file} does not contain any data!!!")
            continue
                
        # Get individual papers
        for arxiv_identifier, paper_data in month_data["papers"].items():
            if not "sbert_encodings" in paper_data:
                print(f"No encodings found for {file} / {arxiv_identifier}")
                continue            
            
            if not "svm" in month_data["papers"][arxiv_identifier]:
                embeddings = month_data["papers"][arxiv_identifier]["sbert_encodings"]
                svm_results = check_svm(embeddings.to(device))
                month_data["papers"][arxiv_identifier]["svm"] = svm_results

        # Store
        with open(file, "wb") as f:
            pickle.dump(month_data, f)
    except Exception as e:
        print(f"Could not process {file}: {str(e)}")
"""

'\n# Iterate over files and generate SVM results\nfrom local_utilities.svm import *\n\nfor i in tqdm(range(len(month_files))):\n    file = month_files[i]\n    # Load file\n    try:\n        with open(file, "rb") as f:\n            month_data = pickle.load(f)\n\n        # Check if there are papers\n        if not "papers" in month_data:\n            print(f"{file} does not contain any data!!!")\n            continue\n\n        # Get individual papers\n        for arxiv_identifier, paper_data in month_data["papers"].items():\n            if not "sbert_encodings" in paper_data:\n                print(f"No encodings found for {file} / {arxiv_identifier}")\n                continue            \n\n            if not "svm" in month_data["papers"][arxiv_identifier]:\n                embeddings = month_data["papers"][arxiv_identifier]["sbert_encodings"]\n                svm_results = check_svm(embeddings.to(device))\n                month_data["papers"][arxiv_identifier]["svm"] = svm_results\n\

In [ ]:
# Iterate over files and generate roberta results
from local_utilities.roberta import *
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm

tokenizer = AutoTokenizer.from_pretrained(get_roberta_directory())
model = AutoModelForSequenceClassification.from_pretrained(get_roberta_directory())
model.to(device)
model.eval()

for i in tqdm(range(len(month_files))):
    file = month_files[i]
    # Load file
    try:
        # Load original data
        with open(file, "rb") as f:
            clean_data = pickle.load(f)
            
        # Load already stored file if it exists
        stored_data_path = file + ".roberta"
        if os.path.exists(stored_data_path):
            with open(stored_data_path, "rb") as f:
                month_data = pickle.load(f)
                
                # Merge new 'clean' data into month data if there is new data present
                month_data["papers"] = clean_data["papers"] | month_data["papers"]    
            
        # Check if there are papers
        if not "papers" in month_data:
            print(f"{file} does not contain any data!!!")
            continue
        
        processed_papers = 0
        # Get individual papers
        for arxiv_identifier, paper_data in month_data["papers"].items():            
            if not "sentences" in paper_data:
                print(f"No sentences found for {file} / {arxiv_identifier}")
                continue
            
            if len(paper_data["sentences"]) == 0:
                print(f"Empty sentences list found for {file} / {arxiv_identifier}")
                continue
            
            if not "roberta" in month_data["papers"][arxiv_identifier]:
                roberta_results = check_roberta(month_data["papers"][arxiv_identifier]["sentences"], device, tokenizer, model)
                month_data["papers"][arxiv_identifier]["roberta"] = roberta_results
                
            processed_papers += 1
            
        print(f"Processed papers: {processed_papers}")

        # Store
        with open(stored_data_path, "wb") as f:
            pickle.dump(month_data, f)
    except Exception as e:
        print(f"Could not process {file}: {str(e)}")

  0%|          | 0/192 [00:00<?, ?it/s]

Processed papers: 279


100%|██████████| 192/192 [00:45<00:00,  4.22it/s]


In [19]:
model.to("cpu")

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

In [20]:
torch.cuda.empty_cache()

In [ ]:
# Iterate over files and generate distilbert results
from local_utilities.distilbert import *
from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast
from tqdm import tqdm

tokenizer = DistilBertTokenizerFast.from_pretrained(get_distilbert_directory())
model = DistilBertForSequenceClassification.from_pretrained(get_distilbert_directory())
model.to(device)
model.eval()

for i in tqdm(range(len(month_files))):
    file = month_files[i]
    # Load file
    try:
        # Load original data
        with open(file, "rb") as f:
            clean_data = pickle.load(f)
            if not "papers" in clean_data:
                print(f"{file} does not contain any data!!!")
                continue
            
        # Load already stored month dta if it exists
        month_data = clean_data # fallback
        stored_data_path = file + ".dbert"
        if os.path.exists(stored_data_path):
            with open(stored_data_path, "rb") as f:
                month_data = pickle.load(f)
                
                # Merge clean data into month data
                month_data["papers"] = clean_data["papers"] | month_data["papers"]

        # Check if there are papers
        if not "papers" in month_data:
            print(f"{file} does not contain any data!!!")
            continue
                
        # Get individual papers
        for arxiv_identifier, paper_data in month_data["papers"].items():
            if not "sentences" in paper_data:
                print(f"No sentences found for {file} / {arxiv_identifier}")
                continue            
            
            if len(paper_data["sentences"]) == 0:
                print(f"Empty sentences list found for {file} / {arxiv_identifier}")
                continue
            
            if not "distilbert" in month_data["papers"][arxiv_identifier]:
                distilbert_results = check_distilbert(month_data["papers"][arxiv_identifier]["sentences"], device, tokenizer, model)
                month_data["papers"][arxiv_identifier]["distilbert"] = distilbert_results

        # Store
        with open(stored_data_path, "wb") as f:
            pickle.dump(month_data, f)
    except Exception as e:
        print(f"Could not process {file}: {str(e)}")

100%|██████████| 192/192 [00:38<00:00,  4.98it/s]


In [22]:
torch.cuda.empty_cache()

In [23]:
# Run GBM on the results
import pickle
from sklearn.ensemble import GradientBoostingClassifier
import torch
from tqdm import tqdm
import traceback

with open("./data/gbm", "rb") as f:
    gbm = pickle.load(f)

for i in tqdm(range(len(month_files))):
    file = month_files[i]
    # Load file
    try:        
        # Open ".sbert.svm.ffnn" file
        with open(file + ".sbert.svm.ffnn", "rb") as f:
            month_data_trad = pickle.load(f)
        
        # Open ".roberta" file
        with open(file + ".roberta", "rb") as f:
            month_data_r = pickle.load(f)
        
        # Open ".dbert" file
        with open(file + ".dbert", "rb") as f:
            month_data_d = pickle.load(f)
        
        # Open "normal" file            
        with open(file, "rb") as f:
            month_data = pickle.load(f)
            
        # Check if there are papers
        if not "papers" in month_data:
            print(f"{file} does not contain any data!!!")
            continue
                
        # Get individual papers
        for arxiv_identifier, paper_data in month_data["papers"].items():
            # Combine AI results
            
            # Skip if there is no data
            if not "ffnn" in month_data_trad["papers"][arxiv_identifier]:
                print(f"No data for {file} : {month_data_trad["papers"][arxiv_identifier]}")
                continue
                        
            ffnn_results = month_data_trad["papers"][arxiv_identifier]["ffnn"]
            svm_results = month_data_trad["papers"][arxiv_identifier]["svm"]
            roberta_results = torch.tensor(month_data_r["papers"][arxiv_identifier]["roberta"]).to("cpu")
            distilbert_results = torch.tensor(month_data_d["papers"][arxiv_identifier]["distilbert"]).to("cpu")
            
            gbm_X = []
            for i in range(len(ffnn_results)):
                gbm_X.append([
                    ffnn_results[i][1],
                    svm_results[i][1],
                    roberta_results[i][1],
                    distilbert_results[i][1]
                ])
                
            gbm_y = gbm.predict_proba(gbm_X)
            month_data["papers"][arxiv_identifier]["gbm"] = gbm_y
            month_data["papers"][arxiv_identifier]["lang"] = month_data_trad["papers"][arxiv_identifier]["lang"]
                
        # Store
        with open(file, "wb") as f:
            pickle.dump(month_data, f)
    except Exception as e:
        print(f"Could not process {file}: {str(e)}")

100%|██████████| 192/192 [00:44<00:00,  4.33it/s]


NOTE
----
When using this data, non-English texts (and empty texts) must be filtered out. Also, the data must be limited to 250 datasets/month